# Data collection
Here I am using preprocessed dataset (after cleaning null values and outliers).

In [1]:
import pandas as pd
df = pd.read_parquet("yellow_tripdata_2025-01_clean.parquet")

In [2]:
num_cols = df.select_dtypes(include='number').columns   # collecting column which has datatype number and assing those column names in a variable

In [3]:
df1 = df[(df[num_cols] >= 0).all(axis=1)]   # removing negative value records

In [4]:
df1=df1.sample(100000)  # Get 100,000 record samples instead of all population which has over 3 Million

In [5]:
df1.shape

(100000, 21)

In [6]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID',
       'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee',
       'trip_duration', 'store_and_fwd_flag_Y'],
      dtype='object')

In [7]:
# splitting the dataset independent variable and dependent variable
X_inde=df1[['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee',
        'store_and_fwd_flag_Y']]

y_dep=df1[['trip_duration']]

In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
# Function for best future from independent variable using Select K function
def selectkbest(indep_X, dep_Y, n):
    from sklearn.feature_selection import SelectKBest, f_regression
    
    test = SelectKBest(score_func=f_regression, k=n)
    fit1 = test.fit(indep_X, dep_Y)
    
    selectk_features = fit1.transform(indep_X)
    return selectk_features

In [69]:
kbest=selectkbest(X_inde,y_dep,6)

In [100]:
print (kbest)

[[  1.2  229.   137.    10.7    0.    15.45]
 [  1.1  246.    48.     7.2    2.89  17.34]
 [  1.52 246.   249.    11.4    1.    19.65]
 ...
 [  1.74  68.   142.     9.3    2.81  16.86]
 [  1.2  230.   162.    10.     5.    19.75]
 [  6.28  33.   138.    35.85   9.75  46.65]]


In [70]:
# initiating list variable
acclin=[]
accsvml=[]
accdes=[]
accrf=[]

In [71]:
# Function for standardizing the data and splitting Training set and test set.
def split_scalar(indep_X,dep_Y):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)    
    return X_train, X_test, y_train, y_test

In [72]:
X_train,X_test,y_train,y_test=split_scalar(kbest,y_dep)  # passing selectk output as input here

In [73]:
# Function for r2_score value
def r2_prediction(regressor,X_test,y_test):
    y_pred = regressor.predict(X_test)
    from sklearn.metrics import r2_score
    r2=r2_score(y_test,y_pred)
    return r2

In [74]:
#Function for Linear Regression model
def Linear(X_train,y_train,X_test,y_test):
    from sklearn.linear_model import LinearRegression
    regressor = LinearRegression()
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2 

#Function for SVR model
def svm_linear(X_train,y_train,X_test,y_test):
    from sklearn.svm import SVR
    regressor = SVR(kernel = 'linear')
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2 
    
#Function for Decission Tree model
def Decision(X_train,y_train,X_test,y_test):
    from sklearn.tree import DecisionTreeRegressor
    regressor = DecisionTreeRegressor(random_state = 0)
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2

#Function for Random Forest model
def random(X_train,y_train,X_test,y_test):
    from sklearn.ensemble import RandomForestRegressor
    regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2

In [75]:
r2_lin=Linear(X_train,y_train,X_test,y_test)
acclin.append(r2_lin)

In [76]:
r2_sl=svm_linear(X_train,y_train,X_test,y_test)    
accsvml.append(r2_sl)

In [77]:
r2_d=Decision(X_train,y_train,X_test,y_test)
accdes.append(r2_d)

In [78]:
r2_r=random(X_train,y_train,X_test,y_test)
accrf.append(r2_r)

In [79]:
def rfe_regression(acclog,accsvml,accdes,accrf): 
    import pandas as pd
    dataframe=pd.DataFrame(index=['R2_Score'],columns=['Linear','SVR','Decision','Random'])

    for number,idex in enumerate(dataframe.index):
        dataframe['Linear'][idex]=acclin[number]       
        dataframe['SVR'][idex]=accsvml[number]
        dataframe['Decision'][idex]=accdes[number]
        dataframe['Random'][idex]=accrf[number]
    return dataframe

In [80]:
result=rfe_regression(acclin,accsvml,accdes,accrf)

In [29]:
#5
result

,Linear,SVR,Decision,Random
R2_Score,0.792729,0.670227,0.835353,0.891668


In [81]:
#6
result

,Linear,SVR,Decision,Random
R2_Score,0.792739,0.670242,0.842205,0.899871


In [55]:
#4
result

,Linear,SVR,Decision,Random
R2_Score,0.792694,0.670127,0.85044,0.883836


In [68]:
#3
result

,Linear,SVR,Decision,Random
R2_Score,0.792702,0.670122,0.849868,0.882576


Conculsion: 6 Features and Random Forest model is giving the better r2 score.

In [82]:
from sklearn.pipeline import Pipeline
pipeline=Pipeline([('model',None)])

In [84]:
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

In [85]:
params=[{'model':[LinearRegression()]},
        {'model':[SVR()],
        'model__kernel':['linear']},
        {'model':[DecisionTreeRegressor()],
        'model__random_state':[0]},
        {'model':[RandomForestRegressor()],
        'model__n_estimators':[10],
        'model__random_state':[0],
        }
       ]

In [86]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(estimator=pipeline,
    param_grid=params,
    cv=5,  # Define your cross-validation strategy
    scoring='f1_weighted', # Define the scoring metric
    verbose=3,
    n_jobs=-1, # Utilize all available cores for parallel computation
    refit=True
)

In [87]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,"Pipeline(step...odel', None)])"
,param_grid,"[{'model': [LinearRegression()]}, {'model': [SVR()], 'model__kernel': ['linear']}, ...]"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,fit_intercept,True


In [88]:
grid.best_params_

{'model': LinearRegression()}

In [89]:
y_pred = grid.predict(X_test)

In [90]:
from sklearn.metrics import r2_score
r2=r2_score(y_test,y_pred)

In [91]:
r2

0.792738724166617

In [92]:
from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
regressor.fit(X_train, y_train)


,n_estimators,10
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [93]:
y_pred = regressor.predict(X_test)

In [94]:
r2=r2_score(y_test,y_pred)

In [95]:
r2

0.8998709794385604

Based on above GridSearchCV is not giving good result. Model RandomForest SelectK (Future selection) n value is 6 which is giving good model. We can take this model as best model and create deployment file.

In [97]:
# Saving the best regressor 
import pickle
file_name = "Fleet_Efficiency.sav"
pickle.dump(regressor,open(file_name, 'wb'))

In [98]:
# saving original dataset input variable columns. These are the columns we receive as input from user
train_columns = X_inde.columns
pickle.dump(train_columns, open('columns.sav', 'wb'))